In [1]:
import pandas as pd 
import numpy as np

In [7]:
# Step 1 (fixed): Monthly aggregation (build monthly_pin_level.csv)
import os
import pandas as pd
import numpy as np

# --- paths (adjust if you renamed files) ---
raw_bio     = "data/raw/Aadhaar Biometric Monthly Update Data.csv"
raw_demo    = "data/raw/Aadhar Demographic Updates for Thane.csv"
raw_enroll  = "data/raw/Aadhar Enrollment Dataset for Thane.csv"
out_dir     = "data/processed"
out_file    = os.path.join(out_dir, "monthly_pin_level.csv")

os.makedirs(out_dir, exist_ok=True)

def find_sum_cols(df, keywords):
    cols = []
    for c in df.columns:
        low = c.lower()
        for kw in keywords:
            if kw in low:
                cols.append(c)
                break
    num_cols = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    return num_cols

def safe_read(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing raw file: {path}")
    df = pd.read_csv(path)
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    else:
        dcands = [c for c in df.columns if 'date' in c.lower()]
        if dcands:
            df['date'] = pd.to_datetime(df[dcands[0]], dayfirst=True, errors='coerce')
        else:
            raise KeyError(f"No date column found in {path}; columns: {df.columns.tolist()}")
    if 'pincode' not in df.columns:
        pcands = [c for c in df.columns if 'pin' in c.lower() or 'pincode' in c.lower()]
        if pcands:
            df.rename(columns={pcands[0]: 'pincode'}, inplace=True)
        else:
            raise KeyError(f"No pincode-like column in {path}; columns: {df.columns.tolist()}")
    df['pincode'] = df['pincode'].astype(str).str.strip()
    return df

bio = safe_read(raw_bio)
demo = safe_read(raw_demo)
enroll = safe_read(raw_enroll)

bio_cols = find_sum_cols(bio, ['bio', 'biometric'])
if not bio_cols:
    bio_cols = [c for c in bio.select_dtypes(include=np.number).columns if c not in ('pincode',)]
print("Detected bio columns:", bio_cols)

demo_cols = find_sum_cols(demo, ['demo', 'demographic'])
if not demo_cols:
    demo_cols = [c for c in demo.select_dtypes(include=np.number).columns if c not in ('pincode',)]
print("Detected demo columns:", demo_cols)

enroll_cols = find_sum_cols(enroll, ['age_', 'enroll', 'enrollment'])
if not enroll_cols:
    enroll_cols = [c for c in enroll.select_dtypes(include=np.number).columns if c not in ('pincode',)]
print("Detected enroll columns:", enroll_cols)

bio = bio.assign(bio_updates = bio[bio_cols].sum(axis=1) if bio_cols else 0)
demo = demo.assign(demo_updates = demo[demo_cols].sum(axis=1) if demo_cols else 0)
enroll = enroll.assign(enrollments = enroll[enroll_cols].sum(axis=1) if enroll_cols else 0)

bio_small = bio[['pincode','date','bio_updates']].copy()
demo_small = demo[['pincode','date','demo_updates']].copy()
enroll_small = enroll[['pincode','date','enrollments']].copy()

daily = bio_small.merge(demo_small, on=['pincode','date'], how='outer')
daily = daily.merge(enroll_small, on=['pincode','date'], how='outer')

for c in ('bio_updates','demo_updates','enrollments'):
    if c in daily.columns:
        daily[c] = daily[c].fillna(0).astype(float)
    else:
        daily[c] = 0.0

daily['total_daily'] = daily[['bio_updates','demo_updates','enrollments']].sum(axis=1)
daily['month'] = daily['date'].dt.to_period('M').astype(str)

# --- aggregate to monthly per pincode (no lambda in agg) ---
monthly = daily.groupby(['pincode','month'], as_index=False).agg(
    total_monthly_count = ('total_daily', 'sum'),
    bio_updates = ('bio_updates','sum'),
    demo_updates = ('demo_updates','sum'),
    enrollments = ('enrollments','sum'),
    num_recorded_days = ('date','nunique')
)

# compute count_active_days separately (days where total_daily > 0)
active_days = (
    daily[daily['total_daily']>0]
    .groupby(['pincode','month'])
    .agg(count_active_days=('date', 'nunique'))
    .reset_index()
)

monthly = monthly.merge(active_days, on=['pincode','month'], how='left')
monthly['count_active_days'] = monthly['count_active_days'].fillna(0).astype(int)

for c in ['total_monthly_count','bio_updates','demo_updates','enrollments']:
    monthly[c] = monthly[c].astype(int)

monthly = monthly.sort_values(['pincode','month']).reset_index(drop=True)
monthly.to_csv(out_file, index=False)

print("WROTE:", out_file)
print("Rows (pincode-month):", len(monthly))
print("Unique pincodes:", monthly['pincode'].nunique())
print("Month range:", monthly['month'].min(), "->", monthly['month'].max())
print("\nSample rows:")
print(monthly.head(10).to_string(index=False))


Detected bio columns: ['bio_age_5_17', 'bio_age_17_']
Detected demo columns: ['demo_age_5_17', 'demo_age_17_']
Detected enroll columns: ['age_0_5', 'age_5_17', 'age_18_greater']
WROTE: data/processed\monthly_pin_level.csv
Rows (pincode-month): 982
Unique pincodes: 96
Month range: 2025-03 -> 2026-01

Sample rows:
pincode   month  total_monthly_count  bio_updates  demo_updates  enrollments  num_recorded_days  count_active_days
 400601 2025-03                 1157            0          1157            0                  1                  1
 400601 2025-04                 2111         1224           887            0                  1                  1
 400601 2025-05                 1821         1697             0          124                  7                  7
 400601 2025-06                  922          835             0           87                 17                 17
 400601 2025-07                 1319         1319             0            0                 20                

In [8]:
# Step 2: Baseline statistics per pincode

# ---------- Load monthly data ----------
monthly = pd.read_csv("data/processed/monthly_pin_level.csv")

# Safety checks
assert monthly.isna().sum().sum() == 0, "NaN values found in monthly data"

# ---------- Baseline stats per pincode ----------
baseline = (
    monthly
    .groupby('pincode')
    .agg(
        median_monthly_total = ('total_monthly_count', 'median'),
        q1_monthly_total     = ('total_monthly_count', lambda x: x.quantile(0.25)),
        q3_monthly_total     = ('total_monthly_count', lambda x: x.quantile(0.75)),
        median_bio           = ('bio_updates', 'median'),
        median_demo          = ('demo_updates', 'median'),
        median_enroll        = ('enrollments', 'median'),
        num_months           = ('month', 'nunique'),
        zero_month_ratio     = ('total_monthly_count', lambda x: (x == 0).mean())
    )
    .reset_index()
)

# ---------- IQR & volatility ----------
baseline['iqr_monthly_total'] = (
    baseline['q3_monthly_total'] - baseline['q1_monthly_total']
)

baseline['iqr_ratio'] = (
    baseline['iqr_monthly_total'] /
    baseline['median_monthly_total'].replace(0, 1)
)

# ---------- Safe stream ratios ----------
baseline['bio_demo_ratio'] = (
    baseline['median_bio'] /
    baseline['median_demo'].replace(0, 1)
)

baseline['enroll_demo_ratio'] = (
    baseline['median_enroll'] /
    baseline['median_demo'].replace(0, 1)
)

# ---------- Save ----------
out_path = "data/processed/pincode_baseline_stats.csv"
baseline.to_csv(out_path, index=False)

# ---------- Diagnostics prints ----------
print("STEP 2 COMPLETE")
print("Saved:", out_path)
print("Total pincodes:", len(baseline))

print("\nTop 10 most volatile PINs (highest IQR ratio):")
print(
    baseline
    .sort_values('iqr_ratio', ascending=False)
    .head(10)[['pincode','median_monthly_total','iqr_monthly_total','iqr_ratio']]
    .to_string(index=False)
)

print("\nPINs with limited history (<6 months):")
print(
    baseline[baseline['num_months'] < 6][['pincode','num_months']]
    .to_string(index=False)
)


STEP 2 COMPLETE
Saved: data/processed/pincode_baseline_stats.csv
Total pincodes: 96

Top 10 most volatile PINs (highest IQR ratio):
 pincode  median_monthly_total  iqr_monthly_total  iqr_ratio
  401603                 315.0             1497.0   4.752381
  421401                1133.0             2544.0   2.245366
  401401                  46.0              102.0   2.217391
  401402                  42.0               91.0   2.166667
  401605                 354.0              726.0   2.050847
  421101                 291.0              570.0   1.958763
  401604                 503.0              958.0   1.904573
  421502                  91.0              164.5   1.807692
  401601                  81.0              140.0   1.728395
  401405                  71.0              122.0   1.718310

PINs with limited history (<6 months):
Empty DataFrame
Columns: [pincode, num_months]
Index: []


In [13]:
# Step 3: IQR-based monthly spike detection (FINAL, SAVING FILE)
import pandas as pd

monthly = pd.read_csv("data/processed/monthly_pin_level.csv")
baseline = pd.read_csv("data/processed/pincode_baseline_stats.csv")

df = monthly.merge(
    baseline[['pincode','q3_monthly_total','iqr_monthly_total']],
    on='pincode',
    how='left'
)

# Spike threshold
df['spike_threshold'] = df['q3_monthly_total'] + 3 * df['iqr_monthly_total']

# Spike condition
df['spike_flag'] = df['total_monthly_count'] > df['spike_threshold']

spikes = df[df['spike_flag']].copy()

# Save spike file
out_path = "data/processed/monthly_spike_anomalies.csv"
spikes.to_csv(out_path, index=False)

print("STEP 3 COMPLETE")
print("Total spike candidates:", len(spikes))
print("Saved to:", out_path)

print("\nSample spike candidates:")
print(
    spikes[
        ['pincode','month','total_monthly_count','spike_threshold']
    ].head(10).to_string(index=False)
)


STEP 3 COMPLETE
Total spike candidates: 7
Saved to: data/processed/monthly_spike_anomalies.csv

Sample spike candidates:
 pincode   month  total_monthly_count  spike_threshold
  401104 2025-11                  158           143.00
  401201 2025-12                 1248          1238.00
  401207 2025-11                  350           303.25
  401305 2025-12                 2475          2439.50
  401603 2025-11                 7250          6144.00
  421304 2025-11                   37            30.00
  421402 2025-11                 3495          3148.75


In [10]:
# Step 4: Stream imbalance detection (ratio-based)

# ---------- Load data ----------
monthly = pd.read_csv("data/processed/monthly_pin_level.csv")
baseline = pd.read_csv("data/processed/pincode_baseline_stats.csv")

# ---------- Merge baseline ratios ----------
df = monthly.merge(
    baseline[
        [
            'pincode',
            'bio_demo_ratio',
            'enroll_demo_ratio'
        ]
    ],
    on='pincode',
    how='left'
)

# ---------- Safe epsilon ----------
eps = 1e-9

# ---------- Current ratios ----------
df['current_bio_demo_ratio'] = df['bio_updates'] / (df['demo_updates'] + eps)
df['current_enroll_demo_ratio'] = df['enrollments'] / (df['demo_updates'] + eps)

# ---------- Ratio change factors ----------
df['bio_demo_ratio_change'] = (
    df['current_bio_demo_ratio'] /
    (df['bio_demo_ratio'].replace(0, eps) + eps)
)

df['enroll_demo_ratio_change'] = (
    df['current_enroll_demo_ratio'] /
    (df['enroll_demo_ratio'].replace(0, eps) + eps)
)

# ---------- Imbalance flags ----------
df['bio_demo_imbalance'] = (
    (df['bio_demo_ratio_change'] >= 3) |
    (df['bio_demo_ratio_change'] <= 1/3)
)

df['enroll_demo_imbalance'] = (
    (df['enroll_demo_ratio_change'] >= 3) |
    (df['enroll_demo_ratio_change'] <= 1/3)
)

df['stream_imbalance_flag'] = (
    df['bio_demo_imbalance'] |
    df['enroll_demo_imbalance']
)

# ---------- Save ----------
out_path = "data/processed/monthly_stream_imbalances.csv"
df[df['stream_imbalance_flag']].to_csv(out_path, index=False)

# ---------- Diagnostics ----------
print("STEP 4 COMPLETE")
print("Total stream imbalance rows:", df['stream_imbalance_flag'].sum())

print("\nSample stream imbalance rows:")
print(
    df[df['stream_imbalance_flag']]
    [['pincode','month','bio_demo_ratio_change','enroll_demo_ratio_change']]
    .head(10)
    .to_string(index=False)
)


STEP 4 COMPLETE
Total stream imbalance rows: 891

Sample stream imbalance rows:
 pincode   month  bio_demo_ratio_change  enroll_demo_ratio_change
  400601 2025-03           0.000000e+00              0.000000e+00
  400601 2025-04           6.243800e-01              0.000000e+00
  400601 2025-05           7.678441e+11              8.480460e+11
  400601 2025-06           3.778137e+11              5.950000e+11
  400601 2025-07           5.968099e+11              0.000000e+00
  400601 2025-08           4.257757e+11              0.000000e+00
  400602 2025-03           0.000000e+00              0.000000e+00
  400602 2025-04           4.100000e+10              0.000000e+00
  400602 2025-05           5.792244e+10              0.000000e+00
  400602 2025-06           3.157341e+10              7.687500e+10


In [15]:
# STEP 5 (REFINED): Final high-confidence anomaly selection


# ---------- Load inputs ----------
df = pd.read_csv("data/processed/monthly_stream_imbalances.csv")
baseline = pd.read_csv("data/processed/pincode_baseline_stats.csv")
spikes = pd.read_csv("data/processed/monthly_spike_anomalies.csv")

# ---------- Prepare spike flag ----------
spikes = spikes[['pincode','month']].copy()
spikes['spike_flag'] = True

df = df.merge(spikes, on=['pincode','month'], how='left')
df['spike_flag'] = df['spike_flag'].fillna(False)

df = df.merge(
    baseline[['pincode','median_monthly_total']],
    on='pincode',
    how='left'
)

# ---------- Conditions ----------
cond_spike = df['spike_flag']
cond_stream = df['stream_imbalance_flag']
cond_big_volume = df['total_monthly_count'] >= 2 * df['median_monthly_total']

cond_extreme_ratio = (
    (df['bio_demo_ratio_change'] >= 10) |
    (df['enroll_demo_ratio_change'] >= 10)
)

# ---------- Final selection logic ----------
final_anomalies = df[
    (
        # Case A: spike-based anomalies
        (cond_spike & (cond_stream | cond_big_volume))
    ) |
    (
        # Case B: non-spike but extreme behaviour
        (cond_stream & cond_extreme_ratio & cond_big_volume)
    )
].copy()

# ---------- Reasons ----------
final_anomalies['reasons'] = ""

final_anomalies.loc[cond_spike, 'reasons'] += "volume spike; "
final_anomalies.loc[cond_stream, 'reasons'] += "stream imbalance; "
final_anomalies.loc[cond_extreme_ratio, 'reasons'] += "extreme ratio shift; "
final_anomalies.loc[cond_big_volume, 'reasons'] += "unusually high volume; "

# ---------- Save ----------
out_path = "data/processed/final_anomalies.csv"
final_anomalies.to_csv(out_path, index=False)

# ---------- Summary ----------
print("STEP 5 (REFINED) COMPLETE")
print("Final high-confidence anomalies:", len(final_anomalies))

print("\nSample final anomalies:")
print(
    final_anomalies[
        ['pincode','month','total_monthly_count',
         'median_monthly_total','reasons']
    ].head(10).to_string(index=False)
)


STEP 5 (REFINED) COMPLETE
Final high-confidence anomalies: 29

Sample final anomalies:
 pincode   month  total_monthly_count  median_monthly_total                                                        reasons
  400610 2025-05                 1086                 509.0 stream imbalance; extreme ratio shift; unusually high volume; 
  401104 2025-11                  158                  37.0        volume spike; stream imbalance; unusually high volume; 
  401106 2025-06                 1162                 422.0 stream imbalance; extreme ratio shift; unusually high volume; 
  401201 2025-12                 1248                 386.0        volume spike; stream imbalance; unusually high volume; 
  401206 2025-10                  171                  75.0 stream imbalance; extreme ratio shift; unusually high volume; 
  401206 2025-11                  325                  75.0 stream imbalance; extreme ratio shift; unusually high volume; 
  401207 2025-11                  350               

C:\Users\Naman\AppData\Local\Temp\ipykernel_20600\1975262437.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['spike_flag'] = df['spike_flag'].fillna(False)
